# 🌾 AgriVoice — N-ATLAS Server for Agricultural Voice Bot

This notebook runs the N-ATLAS models on a free GPU and exposes them as an API
that the AgriVoice application can connect to.

**Run this on Kaggle (2x T4 free) or Google Colab (T4 free)**

### What this does:
1. Loads N-ATLAS LLM (chat/reasoning)
2. Loads all 4 ASR models (Hausa, Yoruba, Igbo, English)
3. Serves an OpenAI-compatible API via ngrok
4. Your AgriVoice app connects to this API

In [ ]:
!pip install -q transformers accelerate torch fastapi uvicorn pyngrok librosa soundfile pydantic huggingface_hub gtts

In [ ]:
from huggingface_hub import login
login()  # Enter your HF token (must have accepted NCAIR1/N-ATLaS access)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline as hf_pipeline
from datetime import datetime

# Load LLM
print("Loading N-ATLAS LLM...")
tokenizer = AutoTokenizer.from_pretrained("NCAIR1/N-ATLaS")
model = AutoModelForCausalLM.from_pretrained(
    "NCAIR1/N-ATLaS",
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

# Load ASR Models
ASR_MODELS = {
    "ha": "NCAIR1/Hausa-ASR",
    "yo": "NCAIR1/Yoruba-ASR",
    "ig": "NCAIR1/Igbo-ASR",
    "en": "NCAIR1/NigerianAccentedEnglish"
}

asr_pipelines = {}
for lang, mid in ASR_MODELS.items():
    print(f"Loading {mid}...")
    asr_pipelines[lang] = hf_pipeline("automatic-speech-recognition", model=mid, chunk_length_s=30, device=0)

print("All models loaded!")

In [ ]:
# Agricultural system prompts
AGRI_PROMPTS = {
    "en": """You are AgriVoice, an expert Nigerian agricultural advisor powered by N-ATLAS.
You help smallholder farmers across Nigeria with crop management, pest control,
soil health, and market advice. Always give practical, actionable advice suitable
for small-scale Nigerian farmers. Use simple language. Reference Nigerian crop
varieties and local conditions. If unsure, recommend contacting the local ADP office.
N-ATLAS is an initiative of FMCIDE, powered by Awarri Technologies.""",
    
    "ha": """Kai AgriVoice ne, masanin noma na Najeriya wanda N-ATLAS ke taimakawa.
Kuna taimaka wa kananan manoma a fadin Najeriya da sarrafa amfanin gona, kawar da kwari,
lafiyar kasa, da shawarwarin kasuwa. Koyaushe ba da shawarwari masu amfani ga manoman Najeriya.
Yi amfani da harshe mai sauki. N-ATLAS shiri ne na FMCIDE, Awarri Technologies ne suka gina shi.""",
    
    "yo": """Iwọ ni AgriVoice, amọran ogbin Naijiria ti N-ATLAS n ṣe atilẹyin.
O ṣe iranlọwọ fun awọn agbe kekere ni Naijiria pẹlu iṣakoso irugbin, iṣakoso kokoro,
ilera ile, ati imọran ọja. Nigbagbogbo fun ni imọran to wulo fun awọn agbe kekere.
Lo ede ti o rọrun. N-ATLAS jẹ eto ti FMCIDE, Awarri Technologies lo ṣe.""",
    
    "ig": """Ị bụ AgriVoice, ọkachamara ọrụ ugbo Naịjirịa nke N-ATLAS na-akwado.
Ị na-enyere ndị ọrụ ugbo nta aka na Naịjirịa na nlekọta ihe ọkụkụ, nchịkwa ahụhụ,
ahụike ala, na ndụmọdụ ahịa. Nye ndụmọdụ bara uru maka ndị ọrụ ugbo Naịjirịa.
Jiri asụsụ dị mfe. N-ATLAS bụ mmemme nke FMCIDE, Awarri Technologies rụrụ ya."""
}

def chat(user_message, language="en", context=""):
    lang = language[:2].lower()
    system = AGRI_PROMPTS.get(lang, AGRI_PROMPTS["en"])
    if context:
        system += f"\n\nUse this agricultural knowledge to answer:\n{context}"
    
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_message}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False,
        date_string=datetime.now().strftime("%d %b %Y")
    )
    
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=512, temperature=0.1,
            repetition_penalty=1.12, do_sample=True, use_cache=True
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# Quick test
print(chat("How do I prevent cassava mosaic disease?", language="en"))

In [ ]:
# Start API Server
import threading, time, io
import uvicorn, librosa
from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel
from typing import List, Optional

api = FastAPI(title="AgriVoice N-ATLAS Server")

class Msg(BaseModel):
    role: str
    content: str

class ChatReq(BaseModel):
    model: str = "NCAIR1/N-ATLaS"
    messages: List[Msg]
    temperature: Optional[float] = 0.1
    max_tokens: Optional[int] = 512
    context: Optional[str] = ""

@api.post("/v1/chat/completions")
async def chat_api(req: ChatReq):
    user_msg = req.messages[-1].content
    sys_msg = next((m.content for m in req.messages if m.role == 'system'), None)
    lang = "en"
    # Try to detect language from system prompt
    if sys_msg:
        for l in ["ha", "yo", "ig"]:
            if l in sys_msg.lower():
                lang = l
                break
    resp = chat(user_msg, language=lang, context=req.context or "")
    return {"choices": [{"message": {"role": "assistant", "content": resp}, "finish_reason": "stop"}]}

@api.post("/v1/audio/transcriptions")
async def transcribe(file: UploadFile = File(...), language: str = "ha"):
    contents = await file.read()
    audio, _ = librosa.load(io.BytesIO(contents), sr=16000)
    lang = language[:2].lower()
    if lang not in asr_pipelines:
        lang = "en"
    result = asr_pipelines[lang](audio)
    return {"text": result["text"], "language": lang}

@api.get("/health")
async def health():
    return {"status": "ok", "service": "agrivoice-natlas"}

threading.Thread(target=lambda: uvicorn.run(api, host='0.0.0.0', port=8000), daemon=True).start()
print("API server started on port 8000")

In [ ]:
# Create public URL with ngrok
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_TOKEN")  # Get free at https://ngrok.com

url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"🌾 AgriVoice API is LIVE!")
print(f"   {url}")
print(f"{'='*60}")
print(f"\nSet this in your AgriVoice .env file:")
print(f"  NATLAS_API_URL={url}")
print(f"\nOr in natlas-sdk:")
print(f"  client = NAtlasClient(backend='openai', base_url='{url}/v1')")

In [ ]:
# Keep alive
import time
while True:
    time.sleep(60)
    print(f"[{datetime.now().strftime('%H:%M')}] AgriVoice server alive at {url}")